In [ ]:
import pandas as pd 
df = pd.read_csv('data/glyco_ids.txt', sep='\t')
df = pd.read_csv('data/glyco_ids.txt', sep='\t')

df.fillna(100, inplace=True)
triggered = [x == 'TRUE' for x in df['Triggered'].values]
ratio = [-min(100, x) for x in df['138/144 Ratio'].values]
gly_class = df['Glycan_Class'].values
labels = [x == 'O-linked' for x in gly_class]

print(len(labels), sum(labels))

# Transformer and binned baseline

In [ ]:
import os, random
import contextlib
import gc
from itertools import chain
import pickle

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import ppx
import pandas as pd
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score

import torch
from torch.utils.data import DataLoader
from xgboost import XGBClassifier
from zipfile import ZipFile as zipfile

import lightning as L
import torch.nn.functional as F
from lightning.pytorch.loggers import CSVLogger
from torchmetrics.classification import BinaryAUROC

from depthcharge.transformers import SpectrumTransformerEncoder
from depthcharge.feedforward import FeedForward

import depthcharge as dc
import polars as pl


pos_df = dc.data.spectra_to_df(f'data/train_pos.mgf', ms_level = [2], progress=True)
neg_df = dc.data.spectra_to_df(f'data/train_neg.mgf', ms_level = [2], progress=True)
train = pl.concat([pos_df, neg_df], how="vertical").with_columns(pl.Series(name="label", values=len(pos_df) * [1] + len(neg_df) * [0]))
train_data = dc.data.SpectrumDataset(train.sample(fraction=1.0, shuffle=True),batch_size=32)

pos_df = dc.data.spectra_to_df(f'data/val_pos.mgf', ms_level = [2], progress=True)
neg_df = dc.data.spectra_to_df(f'data/val_neg.mgf', ms_level = [2], progress=True)
valid = pl.concat([pos_df, neg_df], how="vertical").with_columns(pl.Series(name="label", values=len(pos_df) * [1] + len(neg_df) * [0]))
valid_data = dc.data.SpectrumDataset(valid.sample(fraction=1.0, shuffle=False),batch_size=32)

pos_df = dc.data.spectra_to_df(f'data/test_pos.mgf', ms_level = [2], progress=True)
neg_df = dc.data.spectra_to_df(f'data/test_neg.mgf', ms_level = [2], progress=True)
test = pl.concat([pos_df, neg_df], how="vertical").with_columns(pl.Series(name="label", values=len(pos_df) * [1] + len(neg_df) * [0]))
test_data = dc.data.SpectrumDataset(test.sample(fraction=1.0, shuffle=False),batch_size=32)

train_loader = DataLoader(train_data, batch_size=None)
valid_loader = DataLoader(valid_data, batch_size=None)
test_loader = DataLoader(test_data, batch_size=None)

# Train binned embeddings baseline
print("Train binned embeddings baseline")
def bin_spectra(batch, n_bins=100, min_mz=150, max_mz=2000):
    """Bin mass spectra.

    Parameters
    ----------
    batch : dict of torch.Tensor
        The batch of data.
    n_bins : int
        The number of bins.
    min_mz : float
        The lowest m/z bin.
    max_mz : float
        The highest m/z bin.

    Returns
    -------
    torch.Tensor
        The batch of binned mass spectra.
    """
    bins = torch.linspace(min_mz, max_mz, n_bins - 1)
    out = torch.empty(batch["mz_array"].shape[0], n_bins)
    binned = torch.bucketize(batch["mz_array"], bins)
    for i in range(out.shape[0]):
        out[i, :] = torch.bincount(
            binned[i, :], 
            weights=batch["intensity_array"][i, :], 
            minlength=n_bins,
        )

    return out

X_train, y_train = zip(*[(bin_spectra(b), b["label"]) for b in train_loader])
X_train = torch.vstack(X_train).detach().cpu().numpy()
y_train = torch.cat(y_train).detach().cpu().numpy()

X_eval, y_eval = zip(*[(bin_spectra(b), b["label"]) for b in valid_loader])
X_eval = torch.vstack(X_eval).detach().cpu().numpy()
y_eval = torch.cat(y_eval).detach().cpu().numpy()

X_test, y_test = zip(*[(bin_spectra(b), b["label"]) for b in test_loader])
X_test = torch.vstack(X_test).detach().cpu().numpy()
y_test = torch.cat(y_test).detach().cpu().numpy()

binned_xgb = XGBClassifier(n_estimators=1000, eval_metric='auc', early_stopping_rounds=32).fit(X_train, y_train, eval_set=[(X_eval, y_eval)])


# Non-pretrained Depthcharge 
class SpectrumClassifier(SpectrumTransformerEncoder, L.LightningModule):
    """A model for clssifying mass spectra by quality."""
    def __init__(self, *args, **kwargs):
        """Initialize the model."""
        super().__init__(*args, **kwargs)
        self.mz_encoder = dc.encoders.FloatEncoder(self.d_model)
        self.charge_encoder = dc.encoders.FloatEncoder(self.d_model, 1, 10)
        self.head = torch.nn.Sequential(torch.nn.Linear(self.d_model, 1), torch.nn.Sigmoid())
        #(self.d_model, 1, layers=1, append=nn.Sigmoid())
        self.auroc = BinaryAUROC()

    def global_token_hook(self, mz_array, precursor_mz, precursor_charge, *args, **kwargs):
        """Use our cls token"""
        mz_emb = self.mz_encoder(precursor_mz[:, None])
        charge_emb = self.charge_encoder(precursor_charge.type_as(mz_array)[:, None])
        return (mz_emb + charge_emb).squeeze()

    def step(self, batch, step_type):
        """A single step"""
        Y = batch["label"].type_as(batch["mz_array"])
        Y_hat = self.head(self(**batch)[0][:, 0, :]).flatten()
        loss = F.binary_cross_entropy(Y_hat, Y)
        self.log(f"{step_type}_loss", loss.item(), on_step=True, on_epoch=True, prog_bar=True)
        if step_type == "valid":
            self.auroc(Y_hat, Y)
            self.log(f"{step_type}_auroc", self.auroc, on_step=False, on_epoch=True, prog_bar=True)
        return loss
    
    def training_step(self, batch, batch_idx):
        """The training step."""
        return self.step(batch, "train")

    def validation_step(self, batch, batch_idx):
        """The validation step."""
        return self.step(batch, "valid")

    def predict_step(self, batch, batch_idx):
        """The predict step."""
        return self.head(self(**batch)[0][:, 0, :]).flatten()

    def configure_optimizers(self):
        """Configure optimizers for training."""
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        return optimizer

interval = 1000
print("Non-pretrained Depthcharge training")
trainer = L.Trainer(
    max_epochs=100, 
    val_check_interval=interval,
    callbacks=[L.pytorch.callbacks.early_stopping.EarlyStopping(monitor="valid_auroc", mode="max", patience=5)]
)

model = SpectrumClassifier(d_model=512, nhead=16, n_layers=2)
trainer.fit(model, train_loader, valid_loader)
    

In [ ]:
binned_baseline = binned_xgb.predict_proba(X_test)[:,1]
binned_fpr, binned_tpr, _ = sklearn.metrics.roc_curve(y_test, binned_baseline)
print(sklearn.metrics.auc(binned_fpr, binned_tpr))

transformer_baseline = trainer.predict(model,test_loader)
transformer_baseline = np.concatenate(transformer_baseline)
transformer_fpr, transformer_tpr, _ = sklearn.metrics.roc_curve(y_test, transformer_baseline)
print(sklearn.metrics.auc(transformer_fpr, transformer_tpr))

# GlyCounter baseline 

In [26]:
GlyCounterTrain = []
GlyCounterValid = []
GlyCounterTest = []
GlyCounterData = []
ratio = []
for col in df.columns:
    if col[0] == 'X':
        GlyCounterData.append(df[col].values)

GlyCounterData  = np.array(GlyCounterData).T
print(GlyCounterData.shape)

train_idxs = []
valid_idxs = []
test_idxs = []
for i,file in enumerate(df['Spectrum_ScanNumber'].values):
    if int(file.split('_F')[-1]) % 6 == 0:
        test_idxs.append(i)
    elif int(file.split('_F')[-1]) % 6 == 1:
        valid_idxs.append(i)
    else:
        train_idxs.append(i)

GlyCounterTrain = GlyCounterData[train_idxs]
GlyCounterValid = GlyCounterData[valid_idxs]
GlyCounterTest = GlyCounterData[test_idxs]


gly_class = df['Glycan_Class'].values
labels = np.array([x == 'O-linked' for x in gly_class])
y_train = labels[train_idxs]
y_eval = labels[valid_idxs]
y_test = labels[test_idxs]

xgb = XGBClassifier(n_estimators=1000, eval_metric='auc', early_stopping_rounds=32).fit(GlyCounterTrain, y_train, eval_set=[(GlyCounterValid, y_eval)])
glycounter_baseline = xgb.predict_proba(GlyCounterTest)[:,1]

ratio = np.array([-min(100, x) for x in df['138/144 Ratio'].values])[test_idxs]
fpr, tpr, _ = sklearn.metrics.roc_curve(y_test, ratio)
baseline_fpr, baseline_tpr, _ = sklearn.metrics.roc_curve(y_test, glycounter_baseline)



(252970, 54)
[0]	validation_0-auc:0.87991
[1]	validation_0-auc:0.90317
[2]	validation_0-auc:0.91486
[3]	validation_0-auc:0.92211
[4]	validation_0-auc:0.92653
[5]	validation_0-auc:0.92888
[6]	validation_0-auc:0.93017
[7]	validation_0-auc:0.93176
[8]	validation_0-auc:0.93431
[9]	validation_0-auc:0.93500
[10]	validation_0-auc:0.93585
[11]	validation_0-auc:0.93676
[12]	validation_0-auc:0.93732
[13]	validation_0-auc:0.93834
[14]	validation_0-auc:0.93885
[15]	validation_0-auc:0.93904
[16]	validation_0-auc:0.93981
[17]	validation_0-auc:0.94023
[18]	validation_0-auc:0.94065
[19]	validation_0-auc:0.94095
[20]	validation_0-auc:0.94125
[21]	validation_0-auc:0.94146
[22]	validation_0-auc:0.94165
[23]	validation_0-auc:0.94191
[24]	validation_0-auc:0.94189
[25]	validation_0-auc:0.94208
[26]	validation_0-auc:0.94237
[27]	validation_0-auc:0.94257
[28]	validation_0-auc:0.94263
[29]	validation_0-auc:0.94310
[30]	validation_0-auc:0.94344
[31]	validation_0-auc:0.94342
[32]	validation_0-auc:0.94364
[33]	va

# Casanovo foundation

In [ ]:
import os, random
import contextlib
import gc
from itertools import chain
import pickle 

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import ppx
import pandas as pd
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score

import torch
from torch.utils.data import DataLoader
from xgboost import XGBClassifier
from zipfile import ZipFile as zipfile

import lightning as L
import torch.nn.functional as F
from lightning.pytorch.loggers import CSVLogger
from torchmetrics.classification import BinaryAUROC
from torch.utils.data import Dataset, DataLoader

from depthcharge.transformers import SpectrumTransformerEncoder
from depthcharge.feedforward import FeedForward

import depthcharge as dc
import polars as pl

class EmbDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        print(len(data), len(labels))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


class EmbeddingClassifier(L.LightningModule):
    """A model for clssifying mass spectra by quality."""
    def __init__(self, *args, **kwargs):
        """Initialize the model."""
        super().__init__(*args, **kwargs)
        self.head = torch.nn.Sequential(torch.nn.Linear(512, 512), torch.nn.ReLU(), torch.nn.Linear(512, 1), torch.nn.Sigmoid())
        self.auroc = BinaryAUROC()

    def step(self, batch, step_type):
        """A single step"""
        # print(batch)
        # print(batch[0])
        # print(batch[1])
        # print()
        embs = batch[0].type(torch.float)
        Y = batch[1].type(torch.float)
        Y_hat = self.head(embs).flatten()
        loss = F.binary_cross_entropy(Y_hat, Y)
        self.log(f"{step_type}_loss", loss.item(), on_step=True, on_epoch=True, prog_bar=True)
        if step_type == "valid":
            self.auroc(Y_hat, Y)
            self.log(f"{step_type}_auroc", self.auroc, on_step=False, on_epoch=True, prog_bar=True)
        return loss
    
    def training_step(self, batch, batch_idx):
        """The training step."""
        return self.step(batch, "train")

    def validation_step(self, batch, batch_idx):
        """The validation step."""
        return self.step(batch, "valid")

    def predict_step(self, batch, batch_idx):
        """The predict step."""
        embs = batch[0].type(torch.float)
        return self.head(embs).flatten()

    def configure_optimizers(self):
        """Configure optimizers for training."""
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        return optimizer


pos = [emb.numpy() for emb in torch.load(f"embeddings/train_pos.mgf.pt")]
neg = [emb.numpy() for emb in torch.load(f"embeddings/train_neg.mgf.pt")]
train_embs = pos + neg
y_train = [1] * len(pos) + [0] * len(neg)

pos = [emb.numpy() for emb in torch.load(f"embeddings/val_pos.mgf.pt")]
neg = [emb.numpy() for emb in torch.load(f"embeddings/val_neg.mgf.pt")]
valid_embs = pos + neg
y_valid = [1] * len(pos) + [0] * len(neg)

pos = [emb.numpy() for emb in torch.load(f"embeddings/test_pos.mgf.pt")]
neg = [emb.numpy() for emb in torch.load(f"embeddings/test_neg.mgf.pt")]
test_embs = pos + neg
y_test = [1] * len(pos) + [0] * len(neg)


train_dataset = EmbDataset(train_embs, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

valid_dataset = EmbDataset(valid_embs, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

test_dataset = EmbDataset(test_embs, y_test)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

interval = 1000
logger = CSVLogger("logs", "model", version=0)
trainer = L.Trainer(
    max_epochs=100, 
    callbacks=[L.pytorch.callbacks.early_stopping.EarlyStopping(monitor="valid_auroc", mode="max", patience=5)]
)
model = EmbeddingClassifier() # peak_encoder=enc
trainer.fit(model, train_loader, valid_loader)

pred = trainer.predict(model, test_loader)
pred = torch.cat(pred).detach().cpu().numpy()

foundation_fpr, foundation_tpr, _ = sklearn.metrics.roc_curve(y_test, pred)
print(sklearn.metrics.auc(foundation_fpr, foundation_tpr))

# Multi-task casanovo foundation

In [ ]:
import os, random
import contextlib
import gc
from itertools import chain
import pickle 

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import ppx
import pandas as pd
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score

import torch
from torch.utils.data import DataLoader
from xgboost import XGBClassifier
from zipfile import ZipFile as zipfile

import lightning as L
import torch.nn.functional as F
from lightning.pytorch.loggers import CSVLogger
from torchmetrics.classification import BinaryAUROC
from torch.utils.data import Dataset, DataLoader

from depthcharge.transformers import SpectrumTransformerEncoder
from depthcharge.feedforward import FeedForward

import depthcharge as dc
import polars as pl

class EmbDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        print(len(data), len(labels))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


class EmbeddingClassifier(L.LightningModule):
    """A model for clssifying mass spectra by quality."""
    def __init__(self, *args, **kwargs):
        """Initialize the model."""
        super().__init__(*args, **kwargs)
        self.head = torch.nn.Sequential(torch.nn.Linear(512, 1028), torch.nn.ReLU(), torch.nn.Linear(1028, 128), torch.nn.ReLU(), torch.nn.Linear(128, 1), torch.nn.Sigmoid())
        self.auroc = BinaryAUROC()

    def step(self, batch, step_type):
        """A single step"""
        # print(batch)
        # print(batch[0])
        # print(batch[1])
        # print()
        embs = batch[0].type(torch.float)
        Y = batch[1].type(torch.float)
        Y_hat = self.head(embs).flatten()
        loss = F.binary_cross_entropy(Y_hat, Y)
        self.log(f"{step_type}_loss", loss.item(), on_step=True, on_epoch=True, prog_bar=True)
        if step_type == "valid":
            self.auroc(Y_hat, Y)
            self.log(f"{step_type}_auroc", self.auroc, on_step=False, on_epoch=True, prog_bar=True)
        return loss
    
    def training_step(self, batch, batch_idx):
        """The training step."""
        return self.step(batch, "train")

    def validation_step(self, batch, batch_idx):
        """The validation step."""
        return self.step(batch, "valid")

    def predict_step(self, batch, batch_idx):
        """The predict step."""
        embs = batch[0].type(torch.float)
        return self.head(embs).flatten()

    def configure_optimizers(self):
        """Configure optimizers for training."""
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        return optimizer


pos = [emb.numpy() for emb in torch.load(f"embeddings_multi/train_pos.pt")]
neg = [emb.numpy() for emb in torch.load(f"embeddings_multi/train_neg.pt")]
train_embs = pos + neg
y_train = [1] * len(pos) + [0] * len(neg)

pos = [emb.numpy() for emb in torch.load(f"embeddings_multi/test_pos.pt")]
neg = [emb.numpy() for emb in torch.load(f"embeddings_multi/test_neg.pt")]
valid_embs = pos + neg
y_valid = [1] * len(pos) + [0] * len(neg)

train_dataset = EmbDataset(train_embs, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

valid_dataset = EmbDataset(valid_embs, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)

interval = 1 
logger = CSVLogger("logs", "model", version=0)
trainer = L.Trainer(
    max_epochs=50, 
    #val_check_interval=interval,
    callbacks=[L.pytorch.callbacks.early_stopping.EarlyStopping(monitor="valid_auroc", mode="max", patience=5)]
)
model = EmbeddingClassifier() # peak_encoder=enc
trainer.fit(model, train_loader, valid_loader)
#trainer.fit(model, train_loader)

pos = [emb.numpy() for emb in torch.load(f"embeddings_multi/test_pos.pt")]
neg = [emb.numpy() for emb in torch.load(f"embeddings_multi/test_neg.pt")]
test_embs = pos + neg
y_test = [1] * len(pos) + [0] * len(neg)

test_dataset = EmbDataset(test_embs, y_test)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

pred = trainer.predict(model, test_loader)
pred = torch.cat(pred).detach().cpu().numpy()

foundation_joint_fpr, foundation_joint_tpr, _ = sklearn.metrics.roc_curve(y_test, pred)
print(sklearn.metrics.auc(foundation_joint_fpr, foundation_joint_tpr))

## Save results

In [13]:
import pickle

# Save roc curves to disk
results = {"ratio":(fpr, tpr), "binned":(binned_fpr, binned_tpr), "transformer":(transformer_fpr, transformer_tpr), "GlyCounter":(baseline_fpr, baseline_tpr), "casanovo":(foundation_fpr, foundation_tpr), "casanovo_joint":(foundation_joint_fpr, foundation_joint_tpr)}

with open("glyco_roc_fpr_tpr.pkl", "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)
